In [ ]:
#| default_exp core

# Core WebDAV Archiving Utilities

> This module provides tools to upload structured datasets to a WebDAV server, along with metadata, manifest, license, and usage information. It is designed to ensure reproducibility, immutability, and long-term traceability for research datasets.


## How to use this module

Most of the time you do not call these functions directly: the `upload-dataset` command (notebook `01_cli`) calls
`archive_dataset`, which does the whole job. The README (`index.ipynb`) explains how to archive a dataset, how to resume
an upload that stopped, and how to check where an upload stands.

The typical long run on a Mac:

```
caffeinate -is upload-dataset \
  --source "/full/path/to/dataset" \
  --label 2025_TRIALNAME_MARINE_MAMMALS --archive-date YYYYMMDD --version-hash xxxxxxxx \
  --title "..." --source_name "..." --date_collected "..." --description "..." \
  --responsible "Andrea Munafo" --username username --email andrea.munafo@unipi.it \
  --webdav https://u-drive.unipi.it/remote.php/webdav --verify \
  2>&1 | tee "upload_TRIALNAME_$(date +%Y%m%d_%H%M).log"
```

Leave out `--archive-date` and `--version-hash` the first time a dataset is uploaded; the run prints the values to use
when resuming. `caffeinate -is` keeps the Mac awake until the command ends, and `tee` keeps a copy of the output.

The notebook is organised in the order a run uses the functions: dataset preparation (hash, manifest, licence,
README, registry row), the HTTP layer (session, names, remote listing, uploads), the upload itself, verification, the
registry, and finally `archive_dataset`, which ties them together.

## Preparing the dataset and its description

These functions work on the local folder and write the small files that are archived next to the data.

`hash_folder(folder_path)` returns the first 8 hexadecimal characters of an MD5 digest of the bytes of every file in the
folder, walked directory by directory with file names sorted within each directory. The hash becomes part of the remote
folder name `LABEL_YYYYMMDD_HASH`. It reads every byte, so it takes time on large datasets (about half an hour for
100 GB). It includes hidden files such as `.DS_Store`, which the upload skips; see "Known issues" in the README before
changing it, because any change alters the folder names computed for existing data.

`generate_manifest(folder_path)` returns a table with the relative path, size in bytes and modification time of every
file, saved on the server as `manifest.csv`. Like `hash_folder`, it currently includes hidden files.

`write_restricted_license(path, contact_email, author_name, year=None)` writes the `LICENSE.txt` used for every archive:
academic, non-commercial use only, with prior written permission. `write_standard_license` writes an MIT licence instead;
it is not used by the upload.

`write_readme(path, metadata, contact_email)` writes `README.txt` from the metadata (title, collection date, source,
responsible person, description) with instructions for requesting access.

`log_upload_to_registry(registry_path, dataset_url, label, metadata, local_hash)` appends one row to the local registry
CSV (`upload_registry.csv` by default), creating it with a header if needed. The upload calls it only once a dataset is
complete, and only if the registry does not already list that folder.

In [ ]:
#| export
import os
import json
import hashlib
import requests
from datetime import datetime
import pandas as pd
from pathlib import Path
from requests.auth import HTTPBasicAuth
from getpass import getpass
from time import time
import re, sys, tempfile, unicodedata
from urllib.parse import quote, unquote, urlparse
from xml.etree import ElementTree as ET
from http.cookiejar import DefaultCookiePolicy
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
#| export
def hash_folder(folder_path):    
    "Return a short MD5 hash representing the contents of a folder, based on its files."
    h = hashlib.md5()
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            fp = os.path.join(root, file)
            with open(fp, 'rb') as f:
                while chunk := f.read(8192):
                    h.update(chunk)
    return h.hexdigest()[:8]

def generate_manifest(folder_path):
    "Generate a dataframe listing all non-hidden files in the folder, with paths, sizes, and timestamps."
    manifest = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            fp = os.path.join(root, file)
            relative_path = os.path.relpath(fp, folder_path)
            size_bytes = os.path.getsize(fp)
            last_modified = datetime.fromtimestamp(os.path.getmtime(fp)).isoformat()
            manifest.append({
                "relative_path": relative_path,
                "size_bytes": size_bytes,
                "last_modified": last_modified
            })
    return pd.DataFrame(manifest)

def write_standard_license(path, license_type="MIT"):
    "Write a standard open-source MIT LICENSE.txt file at the specified path."
    mit_text = """MIT License

Copyright (c) {}

Permission is hereby granted, free of charge, to any person obtaining a copy
...""".format(datetime.today().year)
    Path(path).write_text(mit_text)

def write_restricted_license(path, contact_email: str, author_name: str, year=None):
    "Write a restrictive LICENSE.txt file forbidding commercial or unauthorised use."
    if year is None:
        year = datetime.today().year

    license_text = f"""Restricted Academic Use License

This dataset is not public. It is shared solely for academic, non-commercial use,
and only with prior written permission from the dataset creators.

Redistribution, publication, or commercial use of this data is strictly prohibited
without explicit authorization.

To request access or inquire about terms of use, please contact:

{author_name} – {contact_email}

Copyright © University of Pisa and Collaborators, {year}
"""
    Path(path).write_text(license_text)


def write_readme(path, metadata: dict, contact_email: str):
    "Write a README.txt file with dataset description and contact info"
    title = metadata.get("title", "Unnamed Dataset")
    date = metadata.get("date_collected", "Unknown Date")
    description = metadata.get("description", "No description provided.")
    source = metadata.get("source", "Unknown source")
    responsible = metadata.get("responsible", "Unknown researcher")

    readme_text = f"""Dataset: {title}
Date: {date}
Collected by: {source}
Archived by: {responsible}

Description:
{description}

Usage:
This dataset is NOT publicly released. Academic use is permitted only with explicit
permission from the owner.

To request access or usage rights, please contact:
{responsible} – {contact_email}

License: See LICENSE.txt
"""
    Path(path).write_text(readme_text)


def log_upload_to_registry(
    registry_path: str,
    dataset_url: str,
    label: str,
    metadata: dict,
    local_hash: str
):
    """
    "Append an entry to a local CSV file to track uploaded datasets and associated metadata."

    Parameters
    ----------
    registry_path : str
        Path to the CSV registry file.
    dataset_url : str
        The full URL where the dataset was uploaded.
    label : str
        The dataset label used for the upload.
    metadata : dict
        Metadata used in the upload (title, source, etc.).
    local_hash : str
        Short hash of the dataset contents.
    """
    from datetime import datetime
    import csv

    fieldnames = [
        "upload_date", "label", "title", "source", "date_collected",
        "responsible", "description", "hash", "url"
    ]

    entry = {
        "upload_date": datetime.today().isoformat(),
        "label": label,
        "title": metadata.get("title", ""),
        "source": metadata.get("source", ""),
        "date_collected": metadata.get("date_collected", ""),
        "responsible": metadata.get("responsible", ""),
        "description": metadata.get("description", ""),
        "hash": local_hash,
        "url": dataset_url
    }

    registry_file = Path(registry_path)
    is_new = not registry_file.exists()

    with open(registry_file, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow(entry)

    print(f"🗂️ Logged upload to registry: {registry_path}")

## Uploading

A run of `upload_dataset_to_webdav` goes through these steps, printing a line for each:

1. scan the local folder and list the files to upload (hidden files excluded);
2. check the login with one request, so a wrong password fails within seconds;
3. compute the hash, unless `version_hash` is given;
4. list the remote folder, if it exists, to find what is already there and at what size;
5. print the plan, then create missing directories and upload missing files one by one;
6. once every data file is on the server, upload `metadata.json`, `manifest.csv`, `LICENSE.txt` and `README.txt`;
7. if nothing failed, add the row to the local registry.

It is safe to stop at any point and run again: step 4 finds what is already on the server and step 5 sends only the rest.

Guarantees:

- It never issues DELETE, MOVE or any other removal, locally or remotely.
- It never overwrites an existing dataset file unless `overwrite_mismatched=True` (`--overwrite-mismatched`). Every PUT carries `If-None-Match: *`, so the server itself refuses to replace a file that already exists. The one deliberate overwrite is `sync_upload_registry`, which replaces the shared registry CSV with the merged version (the merge keeps every remote row).
- `dry_run` sends read-only requests only (PROPFIND). It never sends MKCOL or PUT.
- A registry entry is written, and the shared registry synced, only when every file and every sidecar is on the server with the right size.

The sections below describe the pieces in the order they are defined.

## HTTP session

One `requests.Session` for the whole run, so TCP and TLS connections are reused instead of opened per request.
urllib3 only retries methods listed in `allowed_methods`; its default set does not include the WebDAV verbs, which is why
the original crash on MKCOL was fatal. All methods listed here are idempotent for our use: dataset PUTs are always sent with
`If-None-Match: *`, so a retried PUT whose first attempt did land gets `412` instead of replacing anything.

With `backoff_factor=2` and 6 retries, urllib3 sleeps roughly 0, 4, 8, 16, 32, 64 s (about two minutes in total) and honours
`Retry-After` on 429/503. That rides out a short rate-limit window. A longer block is handled by the circuit breaker in the upload loop.

The session keeps no cookies. Nextcloud sets session cookies on every response, and a WebDAV request that sends them back without
a CSRF token is rejected with 401. Every request therefore authenticates with Basic auth alone, as the original code did.
A 401 is never retried: it stops the run immediately with a message saying so (see `AuthError`), because repeated failed logins
can trigger Nextcloud's brute-force protection or lock the account.

In [ ]:
#| export
RETRY_METHODS = frozenset(["GET", "HEAD", "PUT", "MKCOL", "PROPFIND"])
RETRY_STATUS = (429, 500, 502, 503, 504)
DEFAULT_TIMEOUT = (15, 300)   # (connect, read) seconds. Read applies per socket operation, not to the whole transfer.
SIDECARS = ("metadata.json", "manifest.csv", "LICENSE.txt", "README.txt")

def make_session(retries: int = 6, backoff_factor: float = 2.0) -> requests.Session:
    "A Session with connection pooling and retry/backoff for the WebDAV verbs."
    retry = Retry(
        total=retries, connect=retries, read=retries, status=retries,
        backoff_factor=backoff_factor,
        status_forcelist=RETRY_STATUS,
        allowed_methods=RETRY_METHODS,
        raise_on_status=False,              # hand the final 5xx back to us instead of raising
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=1, pool_maxsize=2)
    s = requests.Session()
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    # Keep no cookies. Nextcloud answers 401 "CSRF check not passed" to WebDAV requests that carry its
    # session cookie without a CSRF token; the original module-level requests calls never kept cookies.
    s.cookies.set_policy(DefaultCookiePolicy(allowed_domains=[]))
    s.headers["OCS-APIRequest"] = "true"   # also tells Nextcloud not to apply its CSRF check
    return s

## Paths and names

Local relative paths are percent-encoded before going into a URL. The original code passed them raw, which is fine for plain
names but sends `#`, `?` and `%` wrongly (requests treats `#` as the start of a fragment and drops the rest of the name).

Comparisons between local and remote names use Unicode NFC. macOS file names are often NFD, while the server
returns names in its own normal form; without this, a file named `Pisa_è.csv` could be reported missing when it is present.

In [ ]:
#| export
class RemoteError(RuntimeError):
    "A WebDAV request failed in a way that makes continuing unsafe."

class AuthError(RuntimeError):
    "The server rejected the credentials (HTTP 401). Not a RemoteError, so the upload loop never swallows it."

def _dav_message(r) -> str:
    "The readable <s:message> from a Sabre/Nextcloud error body, if there is one."
    try: return (ET.fromstring(r.content).findtext("{http://sabredav.org/ns}message") or "").strip()
    except ET.ParseError: return ""

def _raise_for_auth(r, auth, url: str) -> None:
    "Turn a 401 into an AuthError that says what to check."
    if r.status_code != 401: return
    raise AuthError(
        f"Authentication failed (HTTP 401) for user {getattr(auth, 'username', '?')!r} at {url}\n"
        f"   Server said: {_dav_message(r) or r.reason}\n"
        "   Check the password (a stray space or newline from pasting is a common cause). If u-drive uses\n"
        "   university SSO or two-factor login, create an app password in Settings > Security and use that.\n"
        "   Stopping now rather than retrying: repeated failed logins can trigger Nextcloud's brute-force\n"
        "   protection or lock the account.")

def _nfc(s: str) -> str: return unicodedata.normalize("NFC", s)

def _q(rel: str) -> str:
    "Percent-encode a relative POSIX path for a URL, keeping the separators."
    return quote(rel, safe="/")

def _url_path(url: str) -> str:
    "Decoded, NFC-normalised URL path with duplicate and trailing slashes removed (for comparing hrefs)."
    return _nfc(re.sub("/+", "/", unquote(urlparse(url).path))).rstrip("/")

def _is_not_hidden(path: Path, root: Path) -> bool:
    "Same rule as core.upload_dataset_to_webdav: no path component starts with a dot."
    try: return all(not part.startswith(".") for part in path.relative_to(root).parts)
    except ValueError: return False

PROGRESS_EVERY = 5.0   # seconds between progress lines during long scans and listings

def local_files(source_folder, progress: bool = True) -> dict:
    "Map NFC relative path -> (Path, size) for every non-hidden file, i.e. exactly what gets uploaded."
    root = Path(source_folder)
    out = {}
    t0 = last = time()
    if progress: print(f"🔎 Scanning local files in {root} ...")
    for p in root.rglob("*"):                  # report while walking; sort once at the end
        if p.is_file() and _is_not_hidden(p, root):
            out[_nfc(p.relative_to(root).as_posix())] = (p, p.stat().st_size)
            if progress and time() - last >= PROGRESS_EVERY:
                last = time()
                print(f"   ... {len(out):,} files found so far ({last - t0:.0f} s)")
    return dict(sorted(out.items()))

## Listing the remote tree

`walk_remote` lists a remote folder one directory at a time with `PROPFIND Depth: 1`. For this dataset that is about
315 small requests instead of one `Depth: infinity` response describing 36,240 files (which many servers refuse or time out on).
The same index serves three purposes: resume (which files are already there, and at what size), the directory cache
(which directories already exist, so they are never MKCOL'd again), and verification.

If any listing fails the walk raises instead of returning a partial index. A partial index would make files look missing,
and the upload loop would then try to send them again.

In [ ]:
#| export
_PROPFIND_BODY = (b'<?xml version="1.0" encoding="utf-8"?>'
                  b'<d:propfind xmlns:d="DAV:"><d:prop><d:resourcetype/><d:getcontentlength/></d:prop></d:propfind>')
_NS = "{DAV:}"

def list_remote_dir(session, dir_url: str, auth, timeout=DEFAULT_TIMEOUT):
    "PROPFIND Depth 1. Returns (files {name: size or None}, subdirs [name]), or None if the directory does not exist."
    r = session.request("PROPFIND", dir_url.rstrip("/") + "/", auth=auth, data=_PROPFIND_BODY, timeout=timeout,
                        headers={"Depth": "1", "Content-Type": "application/xml; charset=utf-8"})
    _raise_for_auth(r, auth, dir_url)
    if r.status_code == 404: return None
    if r.status_code != 207: raise RemoteError(f"PROPFIND {dir_url} -> HTTP {r.status_code} {_dav_message(r)}")
    base = _url_path(dir_url)
    files, dirs = {}, []
    for resp in ET.fromstring(r.content).findall(f"{_NS}response"):
        href = resp.findtext(f"{_NS}href")
        if not href: continue
        p = _url_path(href)
        if p == base: continue
        if not p.startswith(base + "/"):
            raise RemoteError(f"Unexpected href {href!r} while listing {dir_url}")
        name = p[len(base) + 1:]
        is_dir, size = False, None
        for ps in resp.findall(f"{_NS}propstat"):
            if " 200 " not in (ps.findtext(f"{_NS}status") or ""): continue
            if ps.find(f"{_NS}prop/{_NS}resourcetype/{_NS}collection") is not None: is_dir = True
            cl = (ps.findtext(f"{_NS}prop/{_NS}getcontentlength") or "").strip()
            if cl.isdigit(): size = int(cl)
        if is_dir: dirs.append(name)
        else: files[name] = size
    return files, dirs

def walk_remote(session, root_url: str, auth, timeout=DEFAULT_TIMEOUT, progress: bool = True):
    "Walk the remote tree. Returns ({rel_file: size or None}, {rel_dir}) or None if root_url does not exist."
    top = list_remote_dir(session, root_url, auth, timeout)
    if top is None: return None
    files, dirs = {}, set()
    pending = [("", top)]
    t0 = last = time()
    while pending:
        rel_dir, (fs, ds) = pending.pop()
        for name, size in fs.items():
            files[f"{rel_dir}/{name}" if rel_dir else name] = size
        for d in ds:
            rel = f"{rel_dir}/{d}" if rel_dir else d
            dirs.add(rel)
            sub = list_remote_dir(session, f"{root_url}/{_q(rel)}", auth, timeout)
            if sub is None: raise RemoteError(f"Directory vanished while listing: {rel}")
            pending.append((rel, sub))
            if progress and time() - last >= PROGRESS_EVERY:
                last = time()
                print(f"   ... {len(dirs) + 1:,} directories listed, {len(files):,} files so far ({last - t0:.0f} s)")
    return files, dirs

def remote_file_size(session, url: str, auth, timeout=DEFAULT_TIMEOUT):
    "HEAD a single file. Returns its size, -1 if present without Content-Length, or None if absent."
    r = session.head(url, auth=auth, timeout=timeout, allow_redirects=True)
    _raise_for_auth(r, auth, url)
    if r.status_code == 404: return None
    if r.status_code not in (200, 204): raise RemoteError(f"HEAD {url} -> HTTP {r.status_code}")
    cl = r.headers.get("Content-Length", "")
    return int(cl) if cl.isdigit() else -1

def check_credentials(session, base_url: str, auth, timeout=DEFAULT_TIMEOUT) -> None:
    "One PROPFIND Depth 0 on the WebDAV root, so bad credentials fail in seconds, before hashing or listing."
    url = base_url.rstrip("/") + "/"
    print(f"🔐 Checking login with {urlparse(url).netloc} ...")
    r = session.request("PROPFIND", url, auth=auth, data=_PROPFIND_BODY, timeout=timeout,
                        headers={"Depth": "0", "Content-Type": "application/xml; charset=utf-8"})
    _raise_for_auth(r, auth, url)
    if r.status_code != 207:
        raise RemoteError(f"WebDAV root {url} -> HTTP {r.status_code} {r.reason} {_dav_message(r)}")
    print(f"🔑 Logged in to {urlparse(url).netloc} as {getattr(auth, 'username', '?')}")

## Creating directories and uploading one file

`ensure_parent_dirs` is the main fix. `known_dirs` is seeded from the remote index and carried across the whole run, so each
directory costs at most one MKCOL for the lifetime of the upload. A directory is only added to the cache once the server
confirms it exists (201, or 405 "already exists"); a failed MKCOL is not cached and is reported for the file that needed it.

`put_file` sends `If-None-Match: *`. If the file already exists the server answers 412 and nothing is replaced. That also covers
the one case urllib3 retries cannot tell apart: a PUT whose response was lost after the server had stored the file. The retry
gets 412, we HEAD the file, and if the size matches we count it as uploaded.

In [ ]:
#| export
def ensure_parent_dirs(session, root_url: str, rel_file: str, auth, known_dirs: set, stats: dict,
                       timeout=DEFAULT_TIMEOUT):
    """MKCOL each missing ancestor of rel_file, top-down, at most once per run.
    rel_file is the local name as-is (used in the URL, as the original code did); the cache key is its NFC form."""
    parts = rel_file.split("/")[:-1]
    for j in range(1, len(parts) + 1):
        rel_dir = "/".join(parts[:j])
        key = _nfc(rel_dir)
        if key in known_dirs: continue
        r = session.request("MKCOL", f"{root_url}/{_q(rel_dir)}", auth=auth, timeout=timeout)
        stats["mkcol"] += 1
        _raise_for_auth(r, auth, root_url)
        if r.status_code in (201, 405): known_dirs.add(key)
        else: raise RemoteError(f"MKCOL {rel_dir} -> HTTP {r.status_code} {r.reason}")

def put_file(session, local_path, url: str, local_size: int, auth, overwrite: bool = False,
             timeout=DEFAULT_TIMEOUT) -> str:
    "Upload one file. Returns 'uploaded', 'present' (already there, same size) or 'exists_different' (not replaced)."
    headers = {} if overwrite else {"If-None-Match": "*"}
    with open(local_path, "rb") as f:
        r = session.put(url, data=f, auth=auth, headers=headers, timeout=timeout)
    _raise_for_auth(r, auth, url)
    if r.status_code in (200, 201, 204): return "uploaded"
    if r.status_code == 412:
        rs = remote_file_size(session, url, auth, timeout)
        return "present" if rs == local_size else "exists_different"
    raise RemoteError(f"PUT -> HTTP {r.status_code} {r.reason}")

def upload_file_to_webdav(local_file_path, remote_url, auth, session=None, overwrite: bool = False,
                          timeout=DEFAULT_TIMEOUT):
    """Upload one local file with PUT and return the response.
    Unlike the original version this never replaces an existing remote file unless overwrite=True
    (the server answers 412 and nothing changes)."""
    if session is None: session = make_session()
    headers = {} if overwrite else {"If-None-Match": "*"}
    with open(local_file_path, "rb") as f:
        r = session.put(remote_url, data=f, auth=auth, headers=headers, timeout=timeout)
    _raise_for_auth(r, auth, remote_url)
    if r.status_code == 412: print(f"⚠️ Already on server, not overwritten: {remote_url}")
    return r

## Upload helpers

`_validate_pins` checks the format of `archive_date` (YYYYMMDD, a real date) and `version_hash` (8 lowercase
hexadecimal characters) before anything is sent. `_fmt_bytes` formats sizes for the progress lines, `_write_log` writes
the failure log, and `_registry_has` checks whether the local registry already lists a folder, which prevents a second
row when a completed upload is run again.

In [ ]:
#| export
def _validate_pins(archive_date, version_hash):
    if archive_date is not None:
        if not re.fullmatch(r"\d{8}", archive_date): raise ValueError(f"archive_date must be YYYYMMDD, got {archive_date!r}")
        datetime.strptime(archive_date, "%Y%m%d")
    if version_hash is not None and not re.fullmatch(r"[0-9a-f]{8}", version_hash):
        raise ValueError(f"version_hash must be 8 lowercase hex characters, got {version_hash!r}")

def _fmt_bytes(n: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB": return f"{n:.1f} {unit}"
        n /= 1024

def _write_log(path: Path, rows) -> None:
    path.write_text("relative_path\tproblem\n" + "".join(f"{p}\t{e}\n" for p, e in rows), encoding="utf-8")

def _registry_has(registry_path, url: str) -> bool:
    p = Path(registry_path)
    return p.exists() and url in p.read_text(encoding="utf-8", errors="replace")

### `upload_dataset_to_webdav`

Uploads, or resumes uploading, one dataset folder and returns a summary dictionary. The keys most often needed are `url`
(the remote folder), `complete` (True only when every file and sidecar is on the server with the right size),
`archive_date` and `version_hash` (the values to pin when resuming), and the counts `present`, `to_upload`, `uploaded`,
`failed` and `mismatched`. `failure_log` gives the path of the failure log when one was written.

Main parameters, in addition to the dataset description:

- `archive_date`, `version_hash`: pin the folder name to resume into an existing archive folder. With `version_hash` given the folder must already exist.
- `dry_run`: list and plan only; nothing is written.
- `overwrite_mismatched`: replace remote files whose size differs from the local file. Off by default.
- `max_consecutive_failures`: stop after this many failures in a row (default 5).
- `timeout`: `(connect, read)` seconds for each request, default `(15, 300)`.
- `registry_path`, `log_dir`: where the local registry row and the failure log are written (default: the current folder).

Compared with the version used before September 2026:

- Credentials are checked with one request before anything else, so a wrong password fails in seconds with a clear
  message instead of after the hashing pass, and a 401 at any point stops the run at once.
- `archive_date` and `version_hash` pin the remote folder name `{label}_{date}_{hash}`. Pinning the hash also skips `hash_folder`,
  which otherwise reads every byte (about half an hour for 105 GB) before anything is sent.
- When `version_hash` is pinned the remote folder must already exist. A mistyped hash then stops the run instead of
  quietly starting a second archive folder.
- Resume works from the remote index, not from a log: a file is skipped when the server has it at the same size. A file that is
  on the server at a *different* size is reported and left alone unless `overwrite_mismatched=True`.
- It returns a summary dict instead of the URL string. Use `res["url"]` where the URL was used before.
- Per-file failures are collected and the loop continues. After `max_consecutive_failures` failures in a row (each one having
  already gone through the retry/backoff cycle) the run stops cleanly: the server is refusing us and pressing on would only add
  load. Re-running the same command resumes.
- Failures and mismatches go to `upload_failures_{folder}_{timestamp}.tsv` in the working directory, next to the registry.
- Sidecars (metadata, manifest, licence, README) are uploaded only once every data file is in place, and are never overwritten.
- The registry entry is written only when there were zero failures and zero mismatches, and only once per remote folder.

In [ ]:
#| export
def upload_dataset_to_webdav(
    source_folder: str,
    base_webdav_url: str,
    label: str,
    metadata: dict,
    username: str,
    contact_email: str,
    license_type: str = "restricted",
    dry_run: bool = False,
    organisation_name: str = "Robotics-Research-Lab",
    archival_data: str = "archival_data",
    auth=None,
    archive_date: str = None,          # YYYYMMDD of the original run, to resume into the same folder
    version_hash: str = None,          # 8-char hash of the original run; also skips re-hashing
    overwrite_mismatched: bool = False,
    max_consecutive_failures: int = 5,
    timeout=DEFAULT_TIMEOUT,
    registry_path: str = "upload_registry.csv",
    log_dir: str = ".",
    session=None,
) -> dict:
    "Upload (or resume uploading) a dataset folder to WebDAV. Returns a summary dict; `complete` is True only if nothing is missing."
    src = Path(source_folder)
    if not src.exists(): raise FileNotFoundError(f"Source folder does not exist: {source_folder}")
    if not src.is_dir(): raise NotADirectoryError(f"Source path is not a directory: {source_folder}")
    _validate_pins(archive_date, version_hash)

    local = local_files(src)
    if not local: raise ValueError(f"Source folder '{source_folder}' contains no uploadable files.")
    total_bytes = sum(s for _, s in local.values())
    print(f"📂 {len(local)} local files, {_fmt_bytes(total_bytes)} (hidden files excluded)")

    if auth is None: auth = HTTPBasicAuth(username, getpass(f"Password for {username}: "))
    if session is None: session = make_session()
    check_credentials(session, base_webdav_url, auth, timeout)   # fail fast, before hashing 100+ GB

    if version_hash is None:
        print("🔢 Hashing folder contents with core.hash_folder (reads every byte; pass version_hash to skip)...")
        t0 = time(); version_hash = hash_folder(source_folder)
        print(f"   hash = {version_hash} ({time() - t0:.0f} s)")
        hash_pinned = False
    else:
        hash_pinned = True
    date_str = archive_date or datetime.today().strftime("%Y%m%d")

    base = base_webdav_url.rstrip("/")
    folder_name = f"{label}_{date_str}_{version_hash}"
    parent_urls = [f"{base}/{quote(organisation_name)}", f"{base}/{quote(organisation_name)}/{quote(archival_data)}"]
    remote_folder_url = f"{parent_urls[-1]}/{quote(folder_name)}"
    print(f"\n📁 Remote folder: {remote_folder_url}")
    print(f"   To resume into this folder later: --archive-date {date_str} --version-hash {version_hash}")

    # ---- index what is already on the server (read-only) ----
    print("🔎 Listing remote folder (PROPFIND Depth 1 per directory)...")
    index = walk_remote(session, remote_folder_url, auth, timeout)
    if index is None:
        if hash_pinned:
            raise RemoteError(f"Remote folder not found: {remote_folder_url}\n"
                              "   A pinned --version-hash means 'resume', so I will not create a new folder. "
                              "Check label, --archive-date and --version-hash against the first run's output.")
        remote_files, known_dirs, root_exists = {}, set(), False
        print("   remote folder does not exist yet; it will be created")
    else:
        remote_files, known_dirs = index
        root_exists = True
        print(f"   remote has {len(remote_files)} files in {len(known_dirs)} subdirectories")

    # ---- plan ----
    todo, present, mismatched = [], [], []
    for rel, (p, size) in local.items():
        if rel in remote_files:
            rsize = remote_files[rel]
            if rsize == size: present.append(rel)
            else: mismatched.append((rel, size, rsize))
        else:
            todo.append(rel)
    if overwrite_mismatched: todo += [m[0] for m in mismatched]
    todo_bytes = sum(local[r][1] for r in todo)
    print(f"\n📋 Plan: {len(present)} already on server (same size), {len(todo)} to upload ({_fmt_bytes(todo_bytes)}), "
          f"{len(mismatched)} on server with a DIFFERENT size"
          + (" (will be overwritten)" if overwrite_mismatched else " (left alone; use --overwrite-mismatched to replace)"))
    for rel, lsz, rsz in mismatched[:20]:
        print(f"   size differs: {rel}  local={lsz}  remote={rsz}")

    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    log_path = Path(log_dir) / f"upload_failures_{folder_name}_{stamp}.tsv"
    summary = dict(url=remote_folder_url, folder=folder_name, archive_date=date_str, version_hash=version_hash,
                   local=len(local), present=len(present), to_upload=len(todo), mismatched=len(mismatched),
                   uploaded=0, failed=0, mkcol=0, aborted=False, complete=False, failure_log=None)

    if dry_run:
        missing_dirs = sorted({"/".join(r.split("/")[:j]) for r in todo
                               for j in range(1, len(r.split("/"))) } - known_dirs)
        if not root_exists: print(f"(dry run) would create folder: {remote_folder_url}")
        print(f"(dry run) would create {len(missing_dirs)} subdirectories (one MKCOL each)")
        for d in missing_dirs[:20]: print(f"   + {d}/")
        if len(missing_dirs) > 20: print(f"   ... and {len(missing_dirs) - 20} more")
        for i, rel in enumerate(todo[:50], 1): print(f"(dry run) [{i}/{len(todo)}] would upload {rel} ({_fmt_bytes(local[rel][1])})")
        if len(todo) > 50: print(f"   ... and {len(todo) - 50} more")
        print("(dry run) no MKCOL or PUT was sent")
        return summary

    # ---- create the dataset folder if needed ----
    if not root_exists:
        for u in parent_urls + [remote_folder_url]:
            r = session.request("MKCOL", u, auth=auth, timeout=timeout)
            summary["mkcol"] += 1
            _raise_for_auth(r, auth, u)
            if r.status_code not in (201, 405): raise RemoteError(f"MKCOL {u} -> HTTP {r.status_code} {r.reason}")
        print(f"📁 Created {remote_folder_url}")

    # ---- upload loop ----
    failures = [(rel, f"size differs on server: local={l} remote={r}") for rel, l, r in mismatched
                if not overwrite_mismatched]
    stats = {"mkcol": 0}
    consecutive = 0
    sent_bytes = 0
    t_start = time()
    for i, rel in enumerate(todo, 1):
        path, size = local[rel]
        rel_orig = path.relative_to(src).as_posix()            # name exactly as on disk, as the first run sent it
        url = f"{remote_folder_url}/{_q(rel_orig)}"
        t0 = time()
        try:
            ensure_parent_dirs(session, remote_folder_url, rel_orig, auth, known_dirs, stats, timeout)
            outcome = put_file(session, path, url, size, auth, overwrite=overwrite_mismatched and rel in remote_files,
                               timeout=timeout)
        except (requests.RequestException, RemoteError, OSError) as e:
            failures.append((rel, f"{type(e).__name__}: {e}"))
            summary["failed"] += 1
            consecutive += 1
            print(f"[{i}/{len(todo)}] ❌ {rel}: {e}")
            if consecutive >= max_consecutive_failures:
                print(f"\n🛑 {consecutive} consecutive failures after retries; stopping. The server is refusing requests. "
                      "Wait, then re-run the same command to resume.")
                summary["aborted"] = True
                break
            continue
        consecutive = 0
        if outcome == "exists_different":
            failures.append((rel, "appeared on server with a different size; not overwritten"))
            summary["failed"] += 1
            print(f"[{i}/{len(todo)}] ⚠️ {rel}: already on server with a different size, not overwritten")
            continue
        summary["uploaded"] += 1
        sent_bytes += size
        el = max(time() - t_start, 1e-9)
        rate = sent_bytes / el
        eta = (todo_bytes - sent_bytes) / rate if rate > 0 else 0
        print(f"[{i}/{len(todo)}] ✅ {rel} ({_fmt_bytes(size)}, {time() - t0:.1f} s)"
              f"{' [was already there]' if outcome == 'present' else ''}  {_fmt_bytes(rate)}/s  ETA {eta / 60:.0f} min")
    summary["mkcol"] += stats["mkcol"]

    data_ok = not failures and not summary["aborted"]

    # ---- sidecars, only once the data is complete ----
    if data_ok:
        metadata = dict(metadata, hash=version_hash, archived_on=date_str)
        print("📝 Preparing metadata.json, manifest.csv (lists every file), LICENSE.txt and README.txt ...")
        with tempfile.TemporaryDirectory() as td:
            td = Path(td)
            (td / "metadata.json").write_text(json.dumps(metadata, indent=4))
            generate_manifest(source_folder).to_csv(td / "manifest.csv", index=False)
            write_restricted_license(td / "LICENSE.txt", contact_email, metadata.get("responsible", "Unknown"))
            write_readme(td / "README.txt", metadata, contact_email)
            print("⬆️ Sidecars: metadata.json, manifest.csv, LICENSE.txt, README.txt")
            for name in SIDECARS:
                if name in remote_files:
                    print(f"   {name}: already on server, not overwritten")
                    continue
                try:
                    out = put_file(session, td / name, f"{remote_folder_url}/{name}", (td / name).stat().st_size,
                                   auth, overwrite=False, timeout=timeout)
                    print(f"   {name}: {out}")
                except (requests.RequestException, RemoteError, OSError) as e:
                    failures.append((name, f"{type(e).__name__}: {e}"))
                    print(f"   {name}: ❌ {e}")
    elif not summary["aborted"]:
        print("⏸️ Sidecars not uploaded: data files are incomplete.")

    summary["complete"] = data_ok and not failures
    dt = (time() - t_start) / 60
    print(f"\n📊 uploaded={summary['uploaded']}  already-there={len(present)}  failed={summary['failed']}  "
          f"size-mismatch={len(mismatched)}  MKCOL={summary['mkcol']}  in {dt:.1f} min")

    if failures:
        _write_log(log_path, failures)
        summary["failure_log"] = str(log_path)
        print(f"   {len(failures)} problems listed in {log_path}")

    if not summary["complete"]:
        print("⚠️ Upload INCOMPLETE; registry not updated. Re-run the same command to resume.")
        return summary

    print("\n✅ Upload complete.")
    if _registry_has(registry_path, remote_folder_url):
        print(f"🗂️ {registry_path} already lists {remote_folder_url}; not adding a duplicate entry.")
    else:
        log_upload_to_registry(registry_path=registry_path, dataset_url=remote_folder_url, label=label,
                               metadata=metadata, local_hash=version_hash)
    return summary

### Legacy: `upload_registry_to_webdav_USEFUL`

Uploads a local registry CSV to a given WebDAV URL with a single PUT, replacing the remote file. It predates
`sync_upload_registry` (below), which merges the local and remote registries before uploading and should be used instead.
Nothing in the package calls this function; it is kept so that old notebooks still run.

In [ ]:
#| export
def upload_registry_to_webdav_USEFUL(
    local_registry_path: str,
    remote_registry_path: str,
    auth: HTTPBasicAuth
):
    "Upload the local registry CSV to a known shared WebDAV location."
    if not os.path.exists(local_registry_path):
        print(f"⚠️ Registry not found at {local_registry_path}")
        return
    with open(local_registry_path, "rb") as f:
        res = requests.put(remote_registry_path, data=f, auth=auth)
    if res.status_code in [200, 201, 204]:
        print(f"📤 Uploaded registry to {remote_registry_path}")
    else:
        print(f"❌ Failed to upload registry: HTTP {res.status_code}")

The next cell repeats imports for the verification and registry code. It is left as it was; the imports at the top of
the notebook already cover everything.

In [ ]:
#| export
from getpass import getpass
import os, json, hashlib, requests
from datetime import datetime
import pandas as pd
from pathlib import Path
from requests.auth import HTTPBasicAuth
from time import time
from xml.etree import ElementTree as ET
import csv

## Verify

`verify_uploaded_dataset(source_folder, remote_folder_url, auth, strict=False)` compares the local folder with the remote
archive folder. It only reads from the server.

It walks the remote folder with `PROPFIND Depth: 1` per directory (via `walk_remote`) and compares names **and sizes**, so a
truncated file is reported under `wrong_size` rather than counted as matched. It also finds files and directories that
exist only on the server, including in directories with no local counterpart.

It prints a summary and returns a dictionary with `matched`, `missing` (local files not on the server) and `extra` (on the
server but not local, sidecars excluded) as sets of relative paths, `wrong_size` as a list of (path, local size, remote
size), and `unknown_size` and `sidecars_missing`. It returns `None` if the remote folder does not exist. With `strict=True`
it raises `AssertionError` if anything is missing, extra or of the wrong size.

`archive_dataset(..., verify=True)` and `upload-dataset --verify` run it automatically after the upload.

In [ ]:
#| export
def verify_uploaded_dataset(
    source_folder: str,
    remote_folder_url: str,
    auth,
    metadata: dict = None,
    base_folder: Path = None,
    strict: bool = False,
    session=None,
    timeout=DEFAULT_TIMEOUT,
) -> dict:
    "Compare the local folder with the remote one by name and size. Read-only."
    if session is None: session = make_session()
    check_credentials(session, remote_folder_url.rstrip("/").rsplit("/", 3)[0], auth, timeout)
    local = {rel: size for rel, (_, size) in local_files(base_folder or source_folder).items()}
    print(f"🔍 Verifying {len(local)} local files against {remote_folder_url}")
    index = walk_remote(session, remote_folder_url, auth, timeout)
    if index is None:
        print("❌ Remote folder does not exist.")
        if strict: raise AssertionError("Remote folder does not exist.")
        return None
    remote, _ = index
    missing = sorted(set(local) - set(remote))
    extra = sorted(set(remote) - set(local) - set(SIDECARS))
    both = set(local) & set(remote)
    wrong_size = sorted((k, local[k], remote[k]) for k in both if remote[k] is not None and remote[k] != local[k])
    unknown_size = sorted(k for k in both if remote[k] is None)
    sidecars_missing = sorted(set(SIDECARS) - set(remote))
    matched = both - {k for k, _, _ in wrong_size} - set(unknown_size)

    print(f"✅ Matched (name and size) : {len(matched)}")
    print(f"❌ Missing on server       : {len(missing)}")
    print(f"📏 Size differs            : {len(wrong_size)}")
    print(f"❔ Size not reported       : {len(unknown_size)}")
    print(f"⚠️  Only on server          : {len(extra)}")
    print(f"📄 Sidecars missing        : {', '.join(sidecars_missing) or 'none'}")
    for f in missing[:40]: print(f"   missing: {f}")
    if len(missing) > 40: print(f"   ... and {len(missing) - 40} more")
    for k, l, r in wrong_size[:40]: print(f"   size: {k}  local={l}  remote={r}")
    for f in extra[:40]: print(f"   only on server: {f}")

    if strict and (missing or extra or wrong_size or unknown_size):
        raise AssertionError("❌ Strict verification failed: mismatches detected.")
    print("\n📦 Verification complete.")
    return {"matched": matched, "missing": set(missing), "extra": set(extra), "wrong_size": wrong_size,
            "unknown_size": set(unknown_size), "sidecars_missing": sidecars_missing}

## Registry

The registry is a CSV file with one row per archived dataset: upload date, label, title, source, collection date,
responsible person, description, hash and URL. There are two copies. The local one (`upload_registry.csv` in the folder
you run from) gets a row when an upload completes. The shared one, on the server at
`Robotics-Research-Lab/archival_data/_registry/upload_registry.csv`, collects the rows from everyone who archives data.

`read_registry_loose` parses a registry whose rows contain unquoted commas in the text fields, by putting the extra
pieces back into the description. `write_registry_quoted` writes a registry with proper CSV quoting, so commas and line
breaks in descriptions are safe.

`sync_upload_registry(local_registry_path, remote_registry_url, auth)` downloads the shared registry, merges it with the
local one, removes duplicate rows (same hash and label, keeping the remote version), saves the result locally as
`upload_registry_merged.csv` and uploads it back, replacing the shared file. Every column is read as text, so hashes that
look like numbers (for example `10739e69` or `00123456`) are kept exactly as written. `archive_dataset` calls it only
after a complete upload. `sync_upload_registry_v0` is the earlier version, kept for reference and not used.

In [ ]:
#| export
# --- Robust CSV handling for registry ---------------------------------------
import csv, io, pandas as pd
from urllib.parse import urlparse, urlunparse

In [ ]:
#| export

EXPECTED_FIELDS = [
    "upload_date","label","title","source","date_collected",
    "responsible","description","hash","url"
]

def read_registry_loose(text: str) -> pd.DataFrame:
    """
    Robustly parse the registry CSV text.
    If a row has >9 columns (unquoted commas in description/title),
    reconstruct by joining the middle segment back into 'description'.
    """
    lines = text.splitlines()
    if not lines:
        return pd.DataFrame(columns=EXPECTED_FIELDS)

    # If the header is wrong or missing, fall back to EXPECTED_FIELDS
    header = [h.strip() for h in lines[0].split(",")]
    if len(header) != 9 or header != EXPECTED_FIELDS:
        header = EXPECTED_FIELDS
        data_lines = lines  # treat all lines as data, we’ll parse flexibly
    else:
        data_lines = lines[1:]

    rows = []
    for ln in data_lines:
        if not ln.strip():
            continue
        parts = [p for p in ln.split(",")]
        # If well-formed: 9 parts
        if len(parts) == 9:
            rows.append(dict(zip(header, parts)))
            continue
        # If malformed with extra commas, assume they belong to description
        if len(parts) > 9:
            rec = {}
            rec["upload_date"]   = parts[0]
            rec["label"]         = parts[1]
            rec["title"]         = parts[2]
            rec["source"]        = parts[3]
            rec["date_collected"]= parts[4]
            rec["responsible"]   = parts[5]
            rec["description"]   = ",".join(parts[6:-2]).strip()
            rec["hash"]          = parts[-2].strip()
            rec["url"]           = parts[-1].strip()
            rows.append(rec)
            continue
        # Too few columns: skip or pad
        # (You can choose to skip; safer than guessing)
        # print(f"Skipping short row: {ln!r}")
        continue

    df = pd.DataFrame(rows, columns=EXPECTED_FIELDS)
    # Strip whitespace
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()
    return df

def write_registry_quoted(df: pd.DataFrame, fields=None) -> str:    
    """
    Write CSV text with proper quoting, ensuring commas in fields are safe.
    """
    fields = fields or EXPECTED_FIELDS
    
    buf = io.StringIO()
    writer = csv.DictWriter(
        buf, fieldnames=EXPECTED_FIELDS,
        quoting=csv.QUOTE_MINIMAL, lineterminator="\n"
    )
    writer.writeheader()
    # for _, row in df.fillna("").iterrows():
    #     writer.writerow({k: row.get(k, "") for k in EXPECTED_FIELDS})
    for record in df.fillna("").to_dict(orient="records"):
        writer.writerow({k: record.get(k, "") for k in fields})    
    return buf.getvalue()

In [ ]:
#| export
def sync_upload_registry_v0(
    local_registry_path: str,
    remote_registry_url: str,
    auth: HTTPBasicAuth,
    merged_local_backup_path: str = "upload_registry_merged.csv"
):
    """
    Safely merge local and remote upload registries, deduplicate, save locally, and upload back to WebDAV.
    Ensures that the _registry folder is created before upload.

    Parameters
    ----------
    local_registry_path : str
        Path to the local registry CSV file.
    remote_registry_url : str
        URL of the remote registry CSV on WebDAV.
    auth : HTTPBasicAuth
        Authentication object for WebDAV access.
    merged_local_backup_path : str
        Path to save the merged registry locally before uploading.
    """
    import io
    from urllib.parse import urlparse, urlunparse

    if not os.path.exists(local_registry_path):
        print(f"⚠️ Local registry does not exist: {local_registry_path}. You should download it first from the server or run `upload_dataset_to_webdav` to upload a new dataset.")
        return

    local_df = pd.read_csv(local_registry_path)

    # Try to download remote registry
    r = requests.get(remote_registry_url, auth=auth)
    if r.status_code == 200:
        remote_df = pd.read_csv(io.StringIO(r.text))
        print(f"🔄 Remote registry downloaded: {len(remote_df)} entries")
    elif r.status_code == 404:
        print("📄 Remote registry not found — creating new one.")
        remote_df = pd.DataFrame(columns=local_df.columns)
    else:
        print(f"❌ Failed to fetch remote registry: HTTP {r.status_code}")
        return

    # Merge and deduplicate
    if not remote_df.empty:
        remote_df = remote_df[[col for col in remote_df.columns if col in local_df.columns]]
    merged_df = pd.concat([local_df, remote_df], ignore_index=True)
    merged_df.drop_duplicates(subset=["hash", "label"], keep="last", inplace=True)

    # Save merged registry locally
    merged_df.to_csv(merged_local_backup_path, index=False)
    print(f"💾 Merged registry saved locally at: {merged_local_backup_path}")

    # Ensure _registry/ folder exists using robust URL parsing
    parsed = urlparse(remote_registry_url)
    registry_folder_path = str(Path(parsed.path).parent)
    registry_folder_url = urlunparse((parsed.scheme, parsed.netloc, registry_folder_path, "", "", ""))

    r_mkcol = requests.request("MKCOL", registry_folder_url, auth=auth)
    if r_mkcol.status_code not in [200, 201, 405]:
        print(f"❌ Failed to create _registry folder: HTTP {r_mkcol.status_code}")
        return

    # Upload the merged registry
    with open(merged_local_backup_path, "rb") as f:
        res = requests.put(remote_registry_url, data=f, auth=auth)

    if res.status_code in [200, 201, 204]:
        print(f"✅ Merged registry uploaded to: {remote_registry_url}")
    else:
        print(f"❌ Failed to upload merged registry: HTTP {res.status_code}")



def sync_upload_registry(
    local_registry_path: str,
    remote_registry_url: str,
    auth: HTTPBasicAuth,
    merged_local_backup_path: str = "upload_registry_merged.csv",
    session=None,
    timeout=DEFAULT_TIMEOUT,
):
    """
    Safely merge local and remote upload registries, deduplicate, save locally, and upload back to WebDAV.
    Handles malformed CSV rows gracefully (extra commas in text fields).
    Ensures that the _registry folder is created before upload.
    Uses the shared retrying session with timeouts. Replaces the remote registry CSV with the merged version.
    """    

    # --- Step 1. Load local registry ---
    if not os.path.exists(local_registry_path):
        print(f"⚠️ Local registry does not exist: {local_registry_path}. You should download it first from the server or run `upload_dataset_to_webdav` to upload a new dataset.")        
        return
    # Read every column as text: otherwise a hash such as 10739e69 is parsed as a float (1.0739e+73)
    # and one such as 00123456 loses its leading zeros when the registry is written back.
    local_df = pd.read_csv(local_registry_path, dtype=str, keep_default_na=False)

    # --- Step 2. Try to download remote registry ---
    if session is None: session = make_session()
    r = session.get(remote_registry_url, auth=auth, timeout=timeout)
    _raise_for_auth(r, auth, remote_registry_url)
    if r.status_code == 200:
        try:
            remote_df = pd.read_csv(io.StringIO(r.text), dtype=str, keep_default_na=False)
        except Exception as e:
            print(f"⚠️ Failed normal CSV parse ({e}); trying tolerant mode.")
            remote_df = read_registry_loose(r.text)
        print(f"🔄 Remote registry downloaded: {len(remote_df)} entries")
    elif r.status_code == 404:
        print("📄 Remote registry not found — creating new one.")
        remote_df = pd.DataFrame(columns=local_df.columns)
    else:
        print(f"❌ Failed to fetch remote registry: HTTP {r.status_code}")
        return

    # --- Step 3. Merge and deduplicate ---
    # if not remote_df.empty:
    #     remote_df = remote_df[[col for col in remote_df.columns if col in local_df.columns]]
    merged_df = pd.concat([local_df, remote_df], ignore_index=True)
    merged_df.drop_duplicates(subset=["hash", "label"], keep="last", inplace=True)
    
    # Enforce column schema and ordering
    for col in EXPECTED_FIELDS:
        if col not in merged_df.columns:
            merged_df[col] = ""
    merged_df = merged_df[EXPECTED_FIELDS]


    # --- Step 4. Save merged registry locally with proper quoting ---
    csv_text = write_registry_quoted(merged_df)
    with open(merged_local_backup_path, "w", encoding="utf-8") as f:
        f.write(csv_text)
    print(f"💾 Merged registry saved locally at: {merged_local_backup_path}")

    # --- Step 5. Ensure remote _registry folder exists ---
    parsed = urlparse(remote_registry_url)
    registry_folder_path = str(Path(parsed.path).parent)
    registry_folder_url = urlunparse((parsed.scheme, parsed.netloc, registry_folder_path, "", "", ""))
    r_mkcol = session.request("MKCOL", registry_folder_url, auth=auth, timeout=timeout)
    if r_mkcol.status_code not in [200, 201, 405]:
        print(f"❌ Failed to create _registry folder: HTTP {r_mkcol.status_code}")
        return

    # --- Step 6. Upload merged registry ---
    with open(merged_local_backup_path, "rb") as f:
        res = session.put(remote_registry_url, data=f, auth=auth, timeout=timeout)

    if res.status_code in [200, 201, 204]:
        print(f"✅ Merged registry uploaded to: {remote_registry_url}")
    else:
        print(f"❌ Failed to upload merged registry: HTTP {res.status_code}")

## Archive a dataset: `archive_dataset`

This is the function behind the `upload-dataset` command. It asks for the password once, then:

1. uploads the dataset with `upload_dataset_to_webdav` (or resumes it, when `archive_date` and `version_hash` are given);
2. verifies the upload with `verify_uploaded_dataset`, if `verify=True`;
3. syncs the shared registry with `sync_upload_registry`, if `sync_registry=True`, a `remote_registry_url` is given and the upload is complete.

The shared registry is synced only when the upload is complete, so an interrupted run never publishes an entry.
`dry_run=True` stops after printing the plan. `archive_dataset` returns the upload summary dictionary described above.

In [ ]:
#| export
def archive_dataset(
    source_folder: str,
    base_webdav_url: str,
    label: str,
    metadata: dict,
    username: str,
    contact_email: str,
    registry_path: str = "upload_registry.csv",
    remote_registry_url: str = None,
    merged_local_backup_path: str = "upload_registry_merged.csv",
    verify: bool = True,
    strict: bool = False,
    sync_registry: bool = True,
    dry_run: bool = False,
    archive_date: str = None,
    version_hash: str = None,
    overwrite_mismatched: bool = False,
    max_consecutive_failures: int = 5,
    read_timeout: float = DEFAULT_TIMEOUT[1],
) -> dict:
    "Upload (resumably), optionally verify, and sync the registry when the upload is complete."
    print("🔐 Requesting credentials...")
    auth = HTTPBasicAuth(username, getpass(f"Password for {username}: "))
    session = make_session()
    timeout = (DEFAULT_TIMEOUT[0], read_timeout)

    summary = upload_dataset_to_webdav(
        source_folder=source_folder, base_webdav_url=base_webdav_url, label=label, metadata=metadata,
        username=username, contact_email=contact_email, dry_run=dry_run, auth=auth, session=session,
        archive_date=archive_date, version_hash=version_hash, overwrite_mismatched=overwrite_mismatched,
        max_consecutive_failures=max_consecutive_failures, timeout=timeout, registry_path=registry_path)

    if verify and not dry_run:
        verify_uploaded_dataset(source_folder=source_folder, remote_folder_url=summary["url"], auth=auth,
                                metadata=metadata, strict=strict, session=session, timeout=timeout)

    if sync_registry and not dry_run and remote_registry_url:
        if summary["complete"]:
            sync_upload_registry(local_registry_path=registry_path, remote_registry_url=remote_registry_url,
                                 auth=auth, merged_local_backup_path=merged_local_backup_path,
                                 session=session, timeout=timeout)
        else:
            print("🗂️ Registry sync skipped: upload is not complete.")
    return summary

## Examples from Python

The cells below show the same workflow step by step from a notebook, using the small test dataset in `data/test_dataset`.
They contact the real server and upload for real, so they are marked `#| eval: false` and are not run by `nbdev_test`.
Run them one at a time, and use a test label so the result does not mix with real archives. For interactive checks of
resuming and verification, see the cells in `02_resumable.ipynb`.

The first example, `archive_dataset`, runs the whole workflow in one call, including the sync of the shared registry,
which adds the test dataset to the list everyone sees. Set `sync_registry=False` there unless that is intended. The
numbered steps after it do the same work one step at a time.

Set your u-drive user name in the next cell.

In [ ]:
my_username = "MY_USERNAME"

In [ ]:
#| eval: false
archive_dataset(
    source_folder="../data/test_dataset",
    base_webdav_url="https://u-drive.unipi.it/remote.php/webdav",    
    label="2025_test_dataset",
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    username=my_username,
    contact_email="andrea.munafo@unipi.it",
    registry_path="upload_registry.csv",
    remote_registry_url="https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/_registry/upload_registry.csv",
    merged_local_backup_path="upload_registry_merged.csv",
    verify=True,
    strict=True,
    sync_registry=True,
    dry_run=False
)

#### 1. Upload the dataset

`upload_dataset_to_webdav` asks for the password, uploads `data/test_dataset` and returns the summary dictionary.
`res["url"]` is the archive folder; `res["archive_date"]` and `res["version_hash"]` are the values to pin if you want to
run it again into the same folder.

In [ ]:
#| eval: false
res = upload_dataset_to_webdav(
    source_folder="../data/test_dataset",
    base_webdav_url="https://u-drive.unipi.it/remote.php/webdav",
    label="2025_test_dataset",
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    username=my_username,
    contact_email="andrea.munafo@unipi.it",
    dry_run=False  # Set to True for a test without uploading
)
remote_url = res["url"]
res

The upload creates the dated, hashed folder on the server, uploads the three test files, then `metadata.json`,
`manifest.csv`, `LICENSE.txt` and `README.txt`, and appends a row to `upload_registry.csv` in the folder the notebook runs
from (`nbs/`).

#### 2. Verify the upload

Compares the local files with the remote folder by name and size. `strict=True` raises an error if anything differs.

In [ ]:
#| eval: false
verify_uploaded_dataset(
    source_folder="../data/test_dataset",
    remote_folder_url=remote_url,
    auth=HTTPBasicAuth(my_username, getpass(f"Password for {my_username}: ")),
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    strict=True
)

#### 3. Synchronise the registry with the server

Merges the local registry with the shared one and uploads the result. This changes the shared registry, so only run it
for datasets that should be listed there.

In [ ]:
#| eval: false
sync_upload_registry(
    local_registry_path="upload_registry.csv",
    remote_registry_url="https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/_registry/upload_registry.csv",
    auth=HTTPBasicAuth(my_username, getpass(f"Password for {my_username}: ")),
    merged_local_backup_path="upload_registry_merged.csv"
)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()